# dialog

> Pure functions that turn a Jupyter notebook (as JSON) into a chat dialog — no JupyterLab, no OpenAI required.

In [ ]:
#| default_exp dialog

In [ ]:
#| hide
from nbdev.showdoc import *

## Tag detection

yasi marks dialog cells with nbdev-style comment tags (e.g. `#| chat_user`). These helpers find those tags in a cell dictionary of a notebook's JSON.

In [ ]:
#| export
def tag_in_cell(cell, # Dictonary of a Jupyter Notebook cell
                tag # The tag to search
               )->bool: # True if any line contains the given tag
    """Checks a Jupyter Notebook cells source, if any line starts with the given tag"""
    return any([line.startswith(tag) for line in cell['source']])

In [ ]:
assert tag_in_cell({'source': ['#| chat_user\n', 'Hi']}, '#| chat_user')
assert not tag_in_cell({'source': ['plain text\n']}, '#| chat_user')

In [ ]:
#| export
def cell_roles(cell, # Dictonary of a Jupyter Notebook cell
               tags:dict # Mapping of role name to tag, e.g. {'user': '#| chat_user'}
              )->list: # Roles whose tag was found in the cell
    """Returns all dialog roles whose tag appears in the given cell"""
    return [role for role, tag in tags.items() if tag_in_cell(cell, tag)]

In [ ]:
cell = {'source': ['#| chat_user\n', '\n', 'How many kangaroos live in Australia?']}
assert cell_roles(cell, {'user': '#| chat_user', 'assistant': '#| chat_assistant'}) == ['user']

## Dialog extraction

`extract_dialog` walks all cells of a notebook and builds the `messages` list for the chat completions API. Cells that carry more than one tag are ambiguous; for those a warning text is returned so the caller can surface it (yasi inserts it as a new markdown cell).

In [ ]:
#| export
def multiple_tags_warning(cell # Dictonary of a Jupyter Notebook cell
                         )->str: # Markdown warning text
    """Builds the warning text for a cell that contains more than one dialog tag"""
    tmp_content = '## \u26a0\u26a0\u26a0 cell contains multiple tags \u26a0\u26a0\u26a0\n'
    tmp_content += 'identify the following cell and select only one tag\n'
    tmp_content += '> ' + '> '.join(cell['source'])
    return tmp_content

In [ ]:
#| export
def extract_dialog(notebook:dict, # Jupyter Notebook JSON as a dictionary
                   tag_system:str='#| chat_system', # tag for system chat markdown cells
                   tag_user:str='#| chat_user', # tag for user chat markdown cells
                   tag_assistant:str='#| chat_assistant' # tag for assistant chat markdown cells
                  )->tuple: # (messages, warnings)
    """Extracts the tagged dialog from a notebook dictionary and turns it into a messages list.
    Returns the messages plus warning texts for cells containing multiple tags."""
    tags = {'system': tag_system, 'user': tag_user, 'assistant': tag_assistant}
    messages, warnings = [], []
    for cell in notebook['cells']:
        roles = cell_roles(cell, tags)
        if len(roles) == 1:
            messages.append({"role": roles[0], "content": ''.join(cell['source'])})
        if len(roles) > 1:
            warnings.append(multiple_tags_warning(cell))
    return messages, warnings

In [ ]:
sample_nb = {'cells': [
    {'cell_type': 'markdown', 'source': ['# Just a heading']},
    {'cell_type': 'markdown', 'source': ['#| chat_system\n', '\n', 'You are a helpful assistant.']},
    {'cell_type': 'markdown', 'source': ['#| chat_user\n', '\n', 'Hello!']},
    {'cell_type': 'markdown', 'source': ['#| chat_assistant\n', '\n', 'Hi, how can I help?']},
    {'cell_type': 'markdown', 'source': ['#| chat_user\n', '#| chat_assistant\n', 'oops, two tags']},
]}
messages, warnings = extract_dialog(sample_nb)
assert [m['role'] for m in messages] == ['system', 'user', 'assistant']
assert messages[1]['content'].endswith('Hello!')
assert len(warnings) == 1 and 'multiple tags' in warnings[0]
messages

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()